<a href="https://colab.research.google.com/github/mariahelenass/Wildfire-Brazil/blob/main/tcc_corre%C3%A7%C3%A3o_do_xgboost_regressor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, make_scorer
from xgboost import XGBRegressor

In [2]:
data = pd.read_csv('/content/drive/MyDrive/focos_qmd_inpe_2024-01-31_2024-12-31_28.863002.csv')

In [3]:
# transformacao para datetime
data["DataHora"] = pd.to_datetime(data["DataHora"])

In [5]:
# pegando ultimios 4 meses do ano
data = data[data["DataHora"].dt.month.isin([8, 9, 10, 11])]

In [6]:
# -999 sao dado invalidos de acordo com a documentacao do inpe
data = data[data["FRP"] != -999]

In [7]:
# -999 sao dado invalidos de acordo com a documentacao do inpe
data = data[data["DiaSemChuva"] != -999]

In [8]:
regiao = {
    "NORTE": ["ACRE", "AMAPÁ", "AMAZONAS", "PARÁ", "RONDÔNIA", "RORAIMA", "TOCANTINS"],
    "NORDESTE": ["ALAGOAS", "BAHIA", "CEARÁ", "MARANHÃO", "PARAÍBA", "PERNAMBUCO", "PIAUÍ", "RIO GRANDE DO NORTE", "SERGIPE"],
    "CENTRO-OESTE": ["GOIÁS", "MATO GROSSO", "MATO GROSSO DO SUL", "DISTRITO FEDERAL"],
    "SUDESTE": ["ESPÍRITO SANTO", "MINAS GERAIS", "RIO DE JANEIRO", "SÃO PAULO"],
    "SUL": ["PARANÁ", "RIO GRANDE DO SUL", "SANTA CATARINA"]
}

In [9]:
def mapear_regiao(estado):
    estado = estado.upper()
    for reg, estados in regiao.items():
        if estado in estados:
            return reg
    return 'DESCONHECIDO'

In [10]:
data['Região'] = data['Estado'].apply(mapear_regiao)

In [11]:
data['Região'].value_counts()

,count
Região,
NORTE,3155889
CENTRO-OESTE,1861627
NORDESTE,906710
SUDESTE,385686
SUL,71845


In [12]:
colunas = ['Pais', 'Municipio', 'Satelite', 'Estado']
df = data.drop(colunas, axis=1)

In [13]:
amostra, _ = train_test_split(df, test_size=0.8, stratify=df['Região'], random_state=42)

print(amostra['Região'].value_counts(normalize=True))

Região
NORTE           0.494518
CENTRO-OESTE    0.291711
NORDESTE        0.142078
SUDESTE         0.060436
SUL             0.011258
Name: proportion, dtype: float64


In [14]:
amostra = amostra.sort_values('DataHora')
amostra = amostra.set_index('DataHora')
amostra['FRP_interpolate'] = amostra['FRP'].interpolate(method='time')
amostra.isnull().sum()

,0
Bioma,0
DiaSemChuva,0
Precipitacao,0
RiscoFogo,0
Latitude,0
Longitude,0
FRP,55554
Região,0
FRP_interpolate,2


In [15]:
amostra['FRP_interpolate'] = amostra['FRP_interpolate'].fillna(method='bfill')
amostra.isnull().sum()

/tmp/ipython-input-15-2005906144.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  amostra['FRP_interpolate'] = amostra['FRP_interpolate'].fillna(method='bfill')


,0
Bioma,0
DiaSemChuva,0
Precipitacao,0
RiscoFogo,0
Latitude,0
Longitude,0
FRP,55554
Região,0
FRP_interpolate,0


In [16]:
# limitando outliers
frp_limite = amostra['FRP_interpolate'].quantile(0.99)
amostra['FRP_tratado'] = amostra['FRP_interpolate'].clip(upper=frp_limite)

In [17]:
amostra = amostra[amostra['FRP_tratado'] >= 0]
amostra['FRP_log'] = np.log1p(amostra['FRP_tratado'])

/tmp/ipython-input-17-2959341146.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  amostra['FRP_log'] = np.log1p(amostra['FRP_tratado'])


In [18]:
prec_limite = amostra['Precipitacao'].quantile(0.99)
amostra['Precipitacao_tratado'] = amostra['Precipitacao'].clip(upper=prec_limite)

In [19]:
amostra['Precipitacao_log'] = np.log1p(amostra['Precipitacao_tratado'])

In [20]:
bioma_dummies = pd.get_dummies(amostra['Bioma'], prefix='Bioma', drop_first=True).astype(int)
regiao_dummies = pd.get_dummies(amostra['Região'], prefix='Região', drop_first=True).astype(int)

amostra = pd.concat([amostra, bioma_dummies, regiao_dummies], axis=1)

In [21]:
amostra = amostra.drop(columns=['Bioma', 'Região'])

In [22]:
# conversao para radianos
lat_rad = np.radians(amostra['Latitude'])
lon_rad = np.radians(amostra['Longitude'])

# Transformando em coordenadas 3D na esfera
amostra['x_coord'] = np.cos(lat_rad) * np.cos(lon_rad)
amostra['y_coord'] = np.cos(lat_rad) * np.sin(lon_rad)
amostra['z_coord'] = np.sin(lat_rad)

In [23]:
cols = ['FRP', 'FRP_interpolate', 'FRP_tratado', 'Precipitacao_log', 'Latitude', 'Longitude']
amostra = amostra.drop(columns=cols, axis=1)

In [28]:
amostra = amostra.reset_index()

In [29]:
# variaveis temporais
amostra['hora'] = amostra['DataHora'].dt.hour
amostra['dia'] = amostra['DataHora'].dt.day
amostra['mes'] = amostra['DataHora'].dt.month
amostra['dia_semana'] = amostra['DataHora'].dt.dayofweek

# transformacao trigonometrica
amostra['hora_sin'] = np.sin(2 * np.pi * amostra['hora'] / 24)
amostra['hora_cos'] = np.cos(2 * np.pi * amostra['hora'] / 24)
amostra['mes_sin'] = np.sin(2 * np.pi * amostra['mes'] / 12)
amostra['mes_cos'] = np.cos(2 * np.pi * amostra['mes'] / 12)
amostra['dia_semana_sin'] = np.sin(2 * np.pi * amostra['dia_semana'] / 7)
amostra['dia_semana_cos'] = np.cos(2 * np.pi * amostra['dia_semana'] / 7)

In [30]:
amostra.head()

,DataHora,DiaSemChuva,Precipitacao,RiscoFogo,FRP_log,Precipitacao_tratado,Bioma_Caatinga,Bioma_Cerrado,Bioma_Mata Atlântica,Bioma_Pampa,...,hora,dia,mes,dia_semana,hora_sin,hora_cos,mes_sin,mes_cos,dia_semana_sin,dia_semana_cos
0,2024-08-01 00:03:16,2.0,0.0,1.00,4.191169,0.0,0,0,1,0,...,0,1,8,3,0.0,1.0,-0.866025,-0.5,0.433884,-0.900969
1,2024-08-01 00:04:51,22.0,0.0,0.59,4.191169,0.0,0,0,0,0,...,0,1,8,3,0.0,1.0,-0.866025,-0.5,0.433884,-0.900969
2,2024-08-01 00:05:20,21.0,0.0,0.20,4.191169,0.0,0,0,0,0,...,0,1,8,3,0.0,1.0,-0.866025,-0.5,0.433884,-0.900969
3,2024-08-01 00:05:30,21.0,0.0,0.82,4.592085,0.0,0,0,0,0,...,0,1,8,3,0.0,1.0,-0.866025,-0.5,0.433884,-0.900969
4,2024-08-01 00:05:38,24.0,0.0,0.81,3.927896,0.0,0,1,0,0,...,0,1,8,3,0.0,1.0,-0.866025,-0.5,0.433884,-0.900969


In [32]:
colunas = ['DiaSemChuva', 'RiscoFogo', 'Precipitacao_tratado']

colunas_extras = ['hora_sin', 'hora_cos','mes_sin', 'mes_cos', 'dia_semana_sin',
                   'dia_semana_cos', 'x_coord', 'y_coord', 'z_coord']

features = colunas + colunas_extras

In [36]:
X = amostra[features].copy()
y = amostra['FRP_log'].copy()

scaler = StandardScaler()
X[features] = scaler.fit_transform(X[features])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scoring = {
    'r2': make_scorer(r2_score),
    'mse': make_scorer(mean_squared_error, greater_is_better=False),
    'mae': make_scorer(mean_absolute_error, greater_is_better=False),
}

model = XGBRegressor(n_jobs=-1, random_state=42)

param_grid = {
    'n_estimators': [200, 500],
    'learning_rate': [0.05, 0.1],
    'max_depth': [8, 12],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=scoring,
    refit='r2',
    cv=cv,
    verbose=1,
    n_jobs=-1,
    return_train_score=False
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\nMelhor combinação de hiperparâmetros:", grid.best_params_)
print(f"R²:  {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Melhor combinação de hiperparâmetros: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 12, 'n_estimators': 500, 'subsample': 0.8}
R²:  0.7202
MSE: 0.5310
MAE: 0.5357
